In [ ]:
# Example 1: Symmetric Encryption with Fernet
from cryptography.fernet import Fernet
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC
import base64
import os

print("Demonstrating symmetric encryption with Fernet...")

# Generate a secure key for encryption
key = Fernet.generate_key()
fernet = Fernet(key)

print(f"✅ Generated encryption key: {key[:20]}...{key[-10:]}")
print(f"✅ Fernet cipher instance created")

# Encrypt some sensitive data
sensitive_data = "This is confidential information: Credit Card 1234-5678-9012-3456"
sensitive_bytes = sensitive_data.encode("utf-8")

encrypted_data = fernet.encrypt(sensitive_bytes)
print(f"✅ Original data length: {len(sensitive_bytes)} bytes")
print(f"✅ Encrypted data length: {len(encrypted_data)} bytes")
print(f"✅ Encrypted data (first 50 chars): {encrypted_data[:50]}...")

# Decrypt the data
decrypted_data = fernet.decrypt(encrypted_data)
decrypted_text = decrypted_data.decode("utf-8")

print(f"✅ Decrypted successfully: {decrypted_text == sensitive_data}")
print(f"✅ Decrypted text: {decrypted_text[:30]}...")

# Password-based key derivation (more realistic scenario)
password = b"my_secure_password_123!"
salt = os.urandom(16)

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=100000,
)
password_key = base64.urlsafe_b64encode(kdf.derive(password))
password_fernet = Fernet(password_key)

# Encrypt with password-derived key
password_encrypted = password_fernet.encrypt(b"Secret message using password-derived key")
password_decrypted = password_fernet.decrypt(password_encrypted)

print(f"✅ Password-based encryption successful")
print(f"✅ Salt length: {len(salt)} bytes")
print(f"✅ KDF iterations: 100,000")
print(f"✅ Password-derived decryption: {password_decrypted.decode()[:30]}...")
print("Example 1: Symmetric encryption with Fernet completed successfully")

In [ ]:
# Example 2: RSA Asymmetric Cryptography and Digital Signatures
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric.utils import Prehashed

print("Demonstrating RSA asymmetric cryptography and digital signatures...")

# Generate RSA key pair
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)
public_key = private_key.public_key()

print(f"✅ Generated RSA key pair (2048-bit)")
print(f"✅ Public exponent: {private_key.public_key().public_numbers().e}")
print(f"✅ Key size: {private_key.key_size} bits")

# Serialize keys for storage/transmission
private_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)

public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM, format=serialization.PublicFormat.SubjectPublicKeyInfo
)

print(f"✅ Private key PEM length: {len(private_pem)} bytes")
print(f"✅ Public key PEM length: {len(public_pem)} bytes")

# RSA encryption/decryption (for small data)
message = b"Secret message for RSA encryption"
encrypted_message = public_key.encrypt(
    message, padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
)

decrypted_message = private_key.decrypt(
    encrypted_message, padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
)

print(f"✅ RSA encryption successful: {decrypted_message == message}")
print(f"✅ Encrypted message length: {len(encrypted_message)} bytes")

# Digital signatures
document = b"Important contract: Terms and conditions apply"
signature = private_key.sign(
    document, padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH), hashes.SHA256()
)

print(f"✅ Digital signature created, length: {len(signature)} bytes")

# Verify signature
try:
    public_key.verify(
        signature,
        document,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
    print("✅ Digital signature verification: VALID")
except Exception:
    print("❌ Digital signature verification: INVALID")
    raise

# Attempt to verify with tampered document
tampered_document = b"Important contract: Modified terms and conditions"
from cryptography.exceptions import InvalidSignature
try:
    public_key.verify(
        signature,
        tampered_document,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
except InvalidSignature:
    print("✅ Tampered document verification: INVALID (correctly detected)")
else:
    raise AssertionError("❌ Tampered document verification: VALID (should be invalid)")

print("Example 2: RSA asymmetric cryptography and digital signatures completed successfully")

In [ ]:
# Example 3: AES Encryption and Cryptographic Hashing
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives import hashes, hmac
from cryptography.hazmat.backends import default_backend
import secrets

print("Demonstrating AES encryption and cryptographic hashing...")


# AES encryption with different modes
def aes_encrypt_decrypt_demo():
    # Generate random key and IV
    key = secrets.token_bytes(32)  # 256-bit key
    iv = secrets.token_bytes(16)  # 128-bit IV

    # Original data to encrypt
    plaintext = b"This is sensitive data that needs AES encryption for protection"

    # Pad data to block size (16 bytes for AES)
    padding_length = 16 - (len(plaintext) % 16)
    padded_plaintext = plaintext + bytes([padding_length] * padding_length)

    # AES-CBC encryption
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    ciphertext = encryptor.update(padded_plaintext) + encryptor.finalize()

    # AES-CBC decryption
    decryptor = cipher.decryptor()
    decrypted_padded = decryptor.update(ciphertext) + decryptor.finalize()

    # Remove padding
    padding_length = decrypted_padded[-1]
    decrypted = decrypted_padded[:-padding_length]

    return {
        "key_length": len(key),
        "iv_length": len(iv),
        "plaintext_length": len(plaintext),
        "ciphertext_length": len(ciphertext),
        "decryption_success": decrypted == plaintext,
        "plaintext": plaintext[:30].decode() + "...",
        "ciphertext_hex": ciphertext[:20].hex() + "...",
    }


# Cryptographic hashing
def crypto_hashing_demo():
    data = b"Important data to hash for integrity verification"

    hash_results = {}

    # SHA-256
    digest_sha256 = hashes.Hash(hashes.SHA256(), backend=default_backend())
    digest_sha256.update(data)
    sha256_hash = digest_sha256.finalize()
    hash_results["SHA256"] = sha256_hash.hex()[:20] + "..."

    # SHA-512
    digest_sha512 = hashes.Hash(hashes.SHA512(), backend=default_backend())
    digest_sha512.update(data)
    sha512_hash = digest_sha512.finalize()
    hash_results["SHA512"] = sha512_hash.hex()[:20] + "..."

    # BLAKE2b
    digest_blake2b = hashes.Hash(hashes.BLAKE2b(64), backend=default_backend())
    digest_blake2b.update(data)
    blake2b_hash = digest_blake2b.finalize()
    hash_results["BLAKE2b"] = blake2b_hash.hex()[:20] + "..."

    return hash_results


# HMAC for message authentication
def hmac_demo():
    key = secrets.token_bytes(32)
    message = b"Authenticated message with HMAC"

    # Create HMAC
    h = hmac.HMAC(key, hashes.SHA256(), backend=default_backend())
    h.update(message)
    mac = h.finalize()

    # Verify HMAC
    h_verify = hmac.HMAC(key, hashes.SHA256(), backend=default_backend())
    h_verify.update(message)

    try:
        h_verify.verify(mac)
        verification_result = "VALID"
    except Exception:
        verification_result = "INVALID"

    return {"mac_length": len(mac), "mac_hex": mac.hex()[:20] + "...", "verification": verification_result}


# Run demonstrations
aes_result = aes_encrypt_decrypt_demo()
print(f"✅ AES-256-CBC encryption:")
print(f"   Key length: {aes_result['key_length']} bytes")
print(f"   Ciphertext length: {aes_result['ciphertext_length']} bytes")
print(f"   Decryption success: {aes_result['decryption_success']}")
print(f"   Ciphertext (hex): {aes_result['ciphertext_hex']}")

hash_results = crypto_hashing_demo()
print(f"✅ Cryptographic hashes:")
for algorithm, hash_value in hash_results.items():
    print(f"   {algorithm}: {hash_value}")

hmac_result = hmac_demo()
print(f"✅ HMAC authentication:")
print(f"   MAC length: {hmac_result['mac_length']} bytes")
print(f"   Verification: {hmac_result['verification']}")
print(f"   MAC (hex): {hmac_result['mac_hex']}")

print("Example 3: AES encryption and cryptographic hashing completed successfully")

In [ ]:
# Example 4: X.509 Certificates and PKI Infrastructure
from cryptography import x509
from cryptography.x509.oid import NameOID, ExtensionOID
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization, hashes
import datetime
import ipaddress

print("Demonstrating X.509 certificates and PKI infrastructure...")

# Generate CA (Certificate Authority) key pair
ca_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# Create self-signed CA certificate
ca_subject = ca_issuer = x509.Name(
    [
        x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
        x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "California"),
        x509.NameAttribute(NameOID.LOCALITY_NAME, "San Francisco"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Demo CA Organization"),
        x509.NameAttribute(NameOID.COMMON_NAME, "Demo Root CA"),
    ]
)

ca_cert = (
    x509.CertificateBuilder()
    .subject_name(ca_subject)
    .issuer_name(ca_issuer)
    .public_key(ca_private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.utcnow())
    .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=365))
    .add_extension(
        x509.BasicConstraints(ca=True, path_length=None),
        critical=True,
    )
    .add_extension(
        x509.KeyUsage(
            digital_signature=True,
            key_cert_sign=True,
            key_encipherment=False,
            data_encipherment=False,
            content_commitment=False,
            key_agreement=False,
            crl_sign=True,
            encipher_only=False,
            decipher_only=False,
        ),
        critical=True,
    )
    .sign(ca_private_key, hashes.SHA256())
)

print(f"✅ Created CA certificate:")
print(f"   Subject: {ca_cert.subject.rfc4514_string()}")
print(f"   Serial: {ca_cert.serial_number}")
print(f"   Valid from: {ca_cert.not_valid_before}")
print(f"   Valid until: {ca_cert.not_valid_after}")

# Generate server certificate key pair
server_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,
)

# Create server certificate signed by CA
server_subject = x509.Name(
    [
        x509.NameAttribute(NameOID.COUNTRY_NAME, "US"),
        x509.NameAttribute(NameOID.STATE_OR_PROVINCE_NAME, "California"),
        x509.NameAttribute(NameOID.LOCALITY_NAME, "San Francisco"),
        x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Demo Server Organization"),
        x509.NameAttribute(NameOID.COMMON_NAME, "demo.example.com"),
    ]
)

server_cert = (
    x509.CertificateBuilder()
    .subject_name(server_subject)
    .issuer_name(ca_cert.subject)
    .public_key(server_private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.utcnow())
    .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=90))
    .add_extension(
        x509.SubjectAlternativeName(
            [
                x509.DNSName("demo.example.com"),
                x509.DNSName("*.demo.example.com"),
                x509.IPAddress(ipaddress.IPv4Address("192.168.1.100")),
            ]
        ),
        critical=False,
    )
    .add_extension(
        x509.KeyUsage(
            digital_signature=True,
            key_encipherment=True,
            key_cert_sign=False,
            data_encipherment=False,
            content_commitment=False,
            key_agreement=False,
            crl_sign=False,
            encipher_only=False,
            decipher_only=False,
        ),
        critical=True,
    )
    .add_extension(
        x509.ExtendedKeyUsage(
            [
                x509.oid.ExtendedKeyUsageOID.SERVER_AUTH,
                x509.oid.ExtendedKeyUsageOID.CLIENT_AUTH,
            ]
        ),
        critical=True,
    )
    .sign(ca_private_key, hashes.SHA256())
)

print(f"✅ Created server certificate:")
print(f"   Subject: {server_cert.subject.rfc4514_string()}")
print(f"   Issuer: {server_cert.issuer.rfc4514_string()}")
print(f"   Serial: {server_cert.serial_number}")

# Certificate validation
print(f"✅ Certificate chain validation:")
print(f"   Server cert signed by CA: {server_cert.issuer == ca_cert.subject}")
print(f"   CA is self-signed: {ca_cert.issuer == ca_cert.subject}")

# Extract certificate information
san_extension = server_cert.extensions.get_extension_for_oid(ExtensionOID.SUBJECT_ALTERNATIVE_NAME)
dns_names = [name.value for name in san_extension.value if isinstance(name, x509.DNSName)]
ip_addresses = [addr.value for addr in san_extension.value if isinstance(addr, x509.IPAddress)]

print(f"✅ Subject Alternative Names:")
print(f"   DNS names: {dns_names}")
print(f"   IP addresses: {[str(ip) for ip in ip_addresses]}")

# Serialize certificates to PEM format
ca_cert_pem = ca_cert.public_bytes(serialization.Encoding.PEM)
server_cert_pem = server_cert.public_bytes(serialization.Encoding.PEM)

print(f"✅ Certificate sizes:")
print(f"   CA certificate PEM: {len(ca_cert_pem)} bytes")
print(f"   Server certificate PEM: {len(server_cert_pem)} bytes")

print("Example 4: X.509 certificates and PKI infrastructure completed successfully")

In [ ]:
# Example 5: Key Exchange and Advanced Cryptographic Protocols
import sys
import traceback

try:
    from cryptography.hazmat.primitives.asymmetric import ec, x25519
    from cryptography.hazmat.primitives.kdf.hkdf import HKDF
    from cryptography.hazmat.primitives.serialization import Encoding, PrivateFormat, PublicFormat, NoEncryption
    from cryptography.hazmat.primitives import hashes
    import os

    print("Demonstrating key exchange and advanced cryptographic protocols...")
    print("\n=== Elliptic Curve Key Exchange (ECDH) ===")

    # Generate EC key pairs for two parties (Alice and Bob)
    alice_private = ec.generate_private_key(ec.SECP384R1())
    alice_public = alice_private.public_key()

    bob_private = ec.generate_private_key(ec.SECP384R1())
    bob_public = bob_private.public_key()

    print(f"✅ Generated ECDH key pairs for Alice and Bob")
    print(f"✅ Curve: SECP384R1")

    # Perform key exchange
    alice_shared = alice_private.exchange(ec.ECDH(), bob_public)
    bob_shared = bob_private.exchange(ec.ECDH(), alice_public)

    print(f"✅ Shared secrets match: {alice_shared == bob_shared}")
    print(f"✅ Shared secret length: {len(alice_shared)} bytes")
    print(f"✅ Shared secret (hex): {alice_shared.hex()[:40]}...")

    print("\n=== X25519 Key Exchange (Modern Curve25519) ===")

    # Modern Curve25519 key exchange
    x25519_alice_private = x25519.X25519PrivateKey.generate()
    x25519_alice_public = x25519_alice_private.public_key()

    x25519_bob_private = x25519.X25519PrivateKey.generate()
    x25519_bob_public = x25519_bob_private.public_key()

    # Perform X25519 key exchange
    x25519_alice_shared = x25519_alice_private.exchange(x25519_bob_public)
    x25519_bob_shared = x25519_bob_private.exchange(x25519_alice_public)

    print(f"✅ X25519 shared secrets match: {x25519_alice_shared == x25519_bob_shared}")
    print(f"✅ X25519 shared secret length: {len(x25519_alice_shared)} bytes")

    print("\n=== Key Derivation Function (HKDF) ===")

    # Derive encryption key from shared secret
    derived_key = HKDF(
        algorithm=hashes.SHA256(), length=32, salt=os.urandom(16), info=b"secure communication session key"
    ).derive(alice_shared)

    print(f"✅ Derived key length: {len(derived_key)} bytes")
    print(f"✅ Derived key (hex): {derived_key.hex()[:40]}...")

    # Derive multiple keys from same shared secret
    master_key = alice_shared
    encryption_key = HKDF(algorithm=hashes.SHA256(), length=32, salt=os.urandom(16), info=b"encryption").derive(
        master_key
    )

    authentication_key = HKDF(algorithm=hashes.SHA256(), length=32, salt=os.urandom(16), info=b"authentication").derive(
        master_key
    )

    print(f"✅ Derived separate encryption and authentication keys")
    print(f"✅ Encryption key (hex): {encryption_key.hex()[:30]}...")
    print(f"✅ Authentication key (hex): {authentication_key.hex()[:30]}...")

    print("\nExample 5: Key exchange and advanced cryptographic protocols completed successfully")

except Exception as e:
    print(f"Example 5: Error in key exchange test: {e}")
    traceback.print_exc()
    raise AssertionError(
        f"cryptography example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e